# Google AI Pro — Colab 200 Compute Units GPU 入门演示

本 Notebook 演示如何最大化利用 Google AI Pro 的开发者权益：
1. 验证每月 **200 Compute Units** 分配的高配 GPU（T4 / L4 / A100）与 PyTorch CUDA 性能
2. 挂载 **5 TB Google Drive** 持久化实验数据与模型权重
3. 使用 Colab Secrets 安全读取 `GEMINI_API_KEY`，调用最新 `google-genai >= 3.5.0` Interactions API (`gemini-4-neon`)
4. 任务完成后自动释放 GPU 运行时，防止空烧 Compute Units

In [ ]:
# 1. 检查当前分配的 GPU 与显存
!nvidia-smi

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")

In [ ]:
# 2. 简易 GPU 矩阵乘基准测试 (FP16)
import time
import torch

if torch.cuda.is_available():
    n = 4096
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    b = torch.randn(n, n, device="cuda", dtype=torch.float16)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        c = a @ b
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / 20
    tflops = (2 * n**3) / dt / 1e12
    print(f"4096x4096 FP16 MatMul: {dt*1000:.2f} ms/iter (~{tflops:.1f} TFLOPS)")
else:
    print("当前为 CPU 运行时。请在右上角 Runtime -> Change runtime type 中切换至 T4 / L4 / A100。")

In [ ]:
# 3. 挂载 5 TB Google Drive（用于持久化数据集、模型与输出）
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. 安装最新 google-genai SDK 并通过 Colab Secrets 调用 Gemini 4 Neon
# 请先在左侧栏 🔑 Secrets 中添加名为 GEMINI_API_KEY 的密钥并开启 Notebook 访问权限
!pip install -q -U "google-genai>=3.5.0"

import os
from google.colab import userdata
from google import genai

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
client = genai.Client()

interaction = client.interactions.create(
    model="gemini-4-neon",
    input="我正在 Google Colab 中使用 Pro 的 200 Compute Units，请给我 3 条最大化 GPU 利用率的实用技巧。"
)
print(interaction.output_text)

In [ ]:
# 5. 【省算力必做】跑完长任务后取消注释下方代码，自动释放 GPU 运行时
# from google.colab import runtime
# runtime.unassign()